In [1]:
%pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

from fastai.vision.all import *
from fastbook import *

Note: you may need to restart the kernel to use updated packages.


# Prepare the data

In [2]:
# Training Set
path = untar_data(URLs.MNIST_SAMPLE)

Path.BASE_PATH = path

threes = (path/'train'/'3').ls().sorted()
sevens = (path/'train'/'7').ls().sorted()

# Load training set into tensors
seven_tensors = [tensor(Image.open(o)) for o in sevens]
three_tensors = [tensor(Image.open(o)) for o in threes]
len(three_tensors),len(seven_tensors)

# stack the tensors (N, 28,28) where N is number of images
# stacking tensors allows for fast calculations to run on all the images at once
stacked_sevens = torch.stack(seven_tensors).float()/255
stacked_threes = torch.stack(three_tensors).float()/255

# Validation set
valid_3_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'3').ls()])
valid_3_tens = valid_3_tens.float()/255

valid_7_tens = torch.stack([tensor(Image.open(o)) for o in (path/'valid'/'7').ls()])
valid_7_tens = valid_7_tens.float()/255

# x is the training set for 3s and 7s, they are all ordered with 3s first then 7s (you can use 3s.length to get the start of 7s)
train_x = torch.cat([stacked_threes, stacked_sevens]).view(-1, 28*28)
# y is the labels on the training set
train_y = tensor([1]*len(threes) + [0]*len(sevens)).unsqueeze(1)

# pair the training set and labels into a dataset
dset = list(zip(train_x,train_y))
x,y = dset[0]

# Validation set
# Zip pairs the images with their labels
valid_x = torch.cat([valid_3_tens, valid_7_tens]).view(-1, 28*28)
valid_y = tensor([1]*len(valid_3_tens) + [0]*len(valid_7_tens)).unsqueeze(1)
valid_dset = list(zip(valid_x,valid_y))

# randomised mini batches of data from the training set.
dl = DataLoader(dset, batch_size=256) # get 256 random images from dset
xb,yb = first(dl) # xb is the data, yb is the labels

valid_dl = DataLoader(valid_dset, batch_size=256) # get 256 random images from valid_dset


In [3]:
class BasicOptim:
    def __init__(self,params,lr): self.params,self.lr = list(params),lr

    # Learning Rate for each parameter
    def step(self, *args, **kwargs):
        for p in self.params: p.data -= p.grad.data * self.lr

    # Reset Gradient to Zero for next batch
    def zero_grad(self, *args, **kwargs):
        for p in self.params: p.grad = None

In [ ]:
# Training 
lr = 1.
linear_model = nn.Linear(28*28,1)
opt = BasicOptim(linear_model.parameters(), lr)

def init_params(size, std=1.0): return (torch.randn(size)*std).requires_grad_()

# Use the sigmoid function to make sure our predictions are between 0 and 1.
def mnist_loss(predictions, targets):
    predictions = predictions.sigmoid()
    return torch.where(targets==1, 1-predictions, predictions).mean()

# gradiant function
def calc_grad(xb, yb, model):
    preds = model(xb)
    loss = mnist_loss(preds, yb)
    loss.backward()

def train_epoch(model):
    for xb,yb in dl:
        calc_grad(xb, yb, model)
        opt.step()
        opt.zero_grad()
# Accuracy not the loss function
def batch_accuracy(xb, yb):
    preds = xb.sigmoid()
    correct = (preds>0.5) == yb
    return correct.float().mean()

def validate_epoch(model):
    accs = [batch_accuracy(model(xb), yb) for xb,yb in valid_dl]
    return round(torch.stack(accs).mean().item(), 4)

def train_model(model, epochs):
    for i in range(epochs):
        train_epoch(model)
        print(validate_epoch(model), end=' ')

In [ ]:
# Training commented out are the parameters, learning rate and optimiser used
# lr = 1.
# linear_model = nn.Linear(28*28,1)
# opt = BasicOptim(linear_model.parameters(), lr)
train_model(linear_model, 20)

0.4932 0.8647 0.8164 0.9092 0.9331 0.9453 0.9541 0.9619 0.9653 0.9668 0.9687 0.9712 0.9731 0.9751 0.9761 0.9761 0.9775 0.9775 0.978 0.9785 

# Fast AI libraries 

In [9]:
linear_model = nn.Linear(28*28,1)
opt = SGD(linear_model.parameters(), lr)
train_model(linear_model, 20)

0.4932 0.8984 0.8066 0.9062 0.9311 0.9463 0.9536 0.9614 0.9643 0.9668 0.9687 0.9712 0.9731 0.9741 0.9751 0.9765 0.9775 0.9785 0.9785 0.9785 

In [6]:

# fast ai using .fit
dls = DataLoaders(dl, valid_dl)
learn = Learner(dls, nn.Linear(28*28,1), opt_func=SGD,
                loss_func=mnist_loss, metrics=batch_accuracy)
learn.fit(10, lr=lr)


epoch,train_loss,valid_loss,batch_accuracy,time
0,0.637283,0.502237,0.495584,00:00
1,0.334614,0.303089,0.694308,00:00
2,0.130343,0.155737,0.858685,00:00
3,0.061674,0.098486,0.917566,00:00
4,0.035972,0.073829,0.934249,00:00
5,0.025642,0.059970,0.950932,00:00
6,0.021212,0.051178,0.956820,00:00
7,0.019115,0.045262,0.963199,00:00
8,0.017965,0.041068,0.966143,00:00
9,0.017221,0.037953,0.969087,00:00


# Non linear

In [ ]:
w1 = init_params((28*28,30))
b1 = init_params(30)
w2 = init_params((30,1))
b2 = init_params(1)

# Model setup (wieghts, bias, relu, gradient tracking)
simple_net = nn.Sequential(
    nn.Linear(28*28,30),
    nn.ReLU(),
    nn.Linear(30,1)
)